# FastSpeech 2 training on Colab (English / Telugu)

Runtime → Change runtime type → **T4 GPU**. Re-running all cells after a disconnect **resumes** from the latest checkpoint on Drive.

One-time upload to `MyDrive/fs2ra/` (made locally, see `docs/ENVIRONMENT.md`):
- `english_features.tar` — `python scripts/package_features.py english`
- `code.zip` — only if the repo is not on GitHub: `git archive -o dist/code.zip HEAD`

Outputs (checkpoints, `train_loss.csv`, `val_loss.csv`, `loss_curves.png`, teacher-forced validation samples) go to `MyDrive/fs2ra/output/<Lang>/`.

In [ ]:
LANG = "English"        # or "Telugu"
REPO_URL = "https://github.com/shishirshetty111/fastspeech2-svarah-telugu.git"  # empty -> MyDrive/fs2ra/code.zip

import os, subprocess
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/fs2ra"
os.makedirs(DRIVE, exist_ok=True)
os.environ["FS2_DATA_ROOT"] = "/content/data"          # features on fast local disk
os.environ["FS2_OUTPUT_ROOT"] = DRIVE + "/output"       # checkpoints + logs survive disconnects
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout or "NO GPU - change runtime type")

In [ ]:
%cd /content
if not os.path.exists("/content/fs2"):
    if REPO_URL:
        !git clone -q $REPO_URL /content/fs2
    else:
        !unzip -q $DRIVE/code.zip -d /content/fs2
%cd /content/fs2
# Training needs only these on top of the Colab image (numpy 2 / torch 2.x verified; no restart needed)
!pip install -q Unidecode==1.4.0 inflect==7.6.0
!python scripts/setup_upstream.py

In [ ]:
lang = LANG.lower()
if not os.path.exists(f"/content/data/{lang}/preprocessed/train.txt"):
    os.makedirs("/content/data", exist_ok=True)
    !cp $DRIVE/{lang}_features.tar /content/
    !tar -xf /content/{lang}_features.tar -C /content/data && rm /content/{lang}_features.tar
!wc -l /content/data/{lang}/preprocessed/train.txt /content/data/{lang}/preprocessed/val.txt

In [ ]:
# Optional: 6-step check of every code path on the GPU (writes to *_dryrun dirs)
!python project/$LANG/train.py --dry-run --total_step 6

In [ ]:
# Full training; resumes automatically from the newest checkpoint in output/<Lang>/ckpt
!python project/$LANG/train.py

In [ ]:
from IPython.display import Image, display
display(Image(f"{DRIVE}/output/{LANG}/log/loss_curves.png"))
!tail -3 $DRIVE/output/$LANG/log/val_loss.csv